## auditのbronzeテーブル作成

In [0]:
%run ../../config

### 空のブロンズテーブルを作成

In [0]:
# 用意したいカラム
# TODO：過去実装からバッククォートで囲む必要がある特殊なカラム名を探す
schema = """
    `event_id` STRING,
    `event_time` STRING,
    `action_name` STRING,
    `resource_name` STRING,
    `source_ip` STRING,
    `user` STRING,
    `request_params` STRING
"""

# Bronzeテーブルを作成
spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {bronze_audit_table_path} (
        {schema},
        _datasource STRING,
        _ingest_timestamp timestamp
    )
    """
)

# # テーブル作り直し
# spark.sql(
#     f"""
#     REPLACE TABLE {bronze_audit_table_path} (
#         {schema},
#         _datasource STRING,
#         _ingest_timestamp timestamp
#     )
#     """
# )



## カラムのズレを修正

原因：値の中にカンマが入っている  
対処法：
```python
    .option("quote", '"')
    .option("escape", '"')  # ダブルクォートをエスケープ
```

In [0]:
from pyspark.sql.functions import col, from_utc_timestamp

# ソースからデータを読み込む
df = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "False")
    .option("quote", '"')
    .option("escape", '"')  # ダブルクォートをエスケープ
    .option("multiLine", "true")  # 複数行にまたがるフィールドがある場合必要
    .load(audit_csv_path)
)

# 監査列として`_datasource`列と`_ingest_timestamp`列を追加
df = (
    df.select("*", "_metadata")
    .withColumn("_datasource", df["_metadata.file_path"])
    .withColumn(
        "_ingest_timestamp",
        from_utc_timestamp(col("_metadata.file_modification_time"), "Asia/Tokyo"),
    )
    .drop("_metadata")
)

In [0]:
df.display()

### bronzeテーブルにdata frameを書き込み

In [0]:
(
    df.write.format("delta")
        .mode("append")
        .saveAsTable(bronze_audit_table_path)
)


In [0]:
# データが書き込まれたことを確認
display(spark.table(bronze_audit_table_path))

### 列追加を検知・追加列を表示したいとき


In [0]:
# # 既存テーブルがあるか確認
# table_exists = spark.catalog.tableExists(bronze_audit_table_path)

# if table_exists:
#     # 既存テーブルのスキーマ取得
#     existing_df = spark.table(bronze_audit_table_path)
#     existing_columns = set(existing_df.columns)

#     # 今回書き込み予定のスキーマ
#     incoming_columns = set(df.columns)

#     # 追加された列を検出（既存にない列）
#     new_columns = incoming_columns - existing_columns

#     if new_columns:
#         raise Exception(
#             f"新しい列が検出されました: {new_columns}\n"
#             "意図しないスキーマ変更の可能性があります。\n"
#             "内容を確認の上、mergeSchemaを明示的に有効化してください。"
#         )

#     # 問題なければ通常書き込み
#     (
#         df.write.format("delta")
#           .mode("append")
#           .saveAsTable(bronze_audit_table_path)
#     )

# else:
#     # 初回作成時
#     (
#         df.write.format("delta")
#           .mode("overwrite")
#           .saveAsTable(bronze_audit_table_path)
#     )


## Deltaテーブルにおけるトランザクション処理の仕組み

![image_1771751938545.png](./image_1771751938545.png "image_1771751938545.png")
[Databricks｜Delta Lake を深堀り：トランザクションログの解析](https://www.databricks.com/jp/blog/2019/08/21/diving-into-delta-lake-unpacking-the-transaction-log.html)より

In [0]:
%sql
describe history my_lab.handson.bronze_audit

In [0]:
%sql
describe detail my_lab.handson.bronze_audit
-- `numFiles`でParquetファイル数が分かる`

Databricks visualization. Run in Databricks to view.